<a href="https://colab.research.google.com/github/MateoGlz/Progra-Analitica-Descriptica-Predictiva/blob/main/28_PracticaReduccionNumerosidad_Mateo_Gonzalez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ==========================================================================
# 📝 Tarea: Reducción de la Numerosidad con CNN, ENN y K-medias usando SVM
# 📊 Dataset: Breast Cancer (Scikit-Learn)
# 👤 Autor/a: [Mateo Gonzalez Lopez]
# 📅 Fecha: [15/11/2025]
# ==========================================================================

## 🔍 Descripción:
En este cuaderno aplicarás técnicas de reducción de la numerosidad (CNN, ENN, K-medias) al conjunto de datos Breast Cancer y analizarás su efecto en el rendimiento de un modelo SVM.

## ==========================================================================

## 1. Cargar librerías necesarias
Escribe aquí la importación de librerías: numpy, pandas, matplotlib, seaborn, sklearn (datasets, model_selection, preprocessing, metrics, svm, kmeans), etc.

En el caso de los códigos de ENN, CNN, estos  deberás tomarlos del cuaderno que contiene la teoría y ejemplos.

In [39]:
#1. Cargar librerías necesarias
!pip install imbalanced-learn
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from sklearn.cluster import KMeans
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense
from imblearn.under_sampling import EditedNearestNeighbours
from sklearn.cluster import KMeans



# --------------------------------------------------------------------------

## 2. Cargar y explorar el conjunto de datos Breast Cancer
- Cargar el dataset con sklearn.datasets.load_breast_cancer
- Explora las dimensiones, variables, y distribución de clases

In [40]:
#2. Cargar y explorar el conjunto de datos Breast Cancer
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [41]:
df['target'] = data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

In [42]:
print("=== INFORMACIÓN BÁSICA DEL DATASET ===")
print(f"Dimensiones del dataset: {df.shape}")
print(f"Número de características: {len(data.feature_names)}")
print(f"Número de muestras: {len(df)}")

print("\n=== DISTRIBUCIÓN DE CLASES ===")
print(df['diagnosis'].value_counts())
print(f"\nPorcentajes:")
print(df['diagnosis'].value_counts(normalize=True) * 100)

print("\n=== INFORMACIÓN DE VARIABLES ===")
df.info()

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(df.describe())

=== INFORMACIÓN BÁSICA DEL DATASET ===
Dimensiones del dataset: (569, 32)
Número de características: 30
Número de muestras: 569

=== DISTRIBUCIÓN DE CLASES ===
diagnosis
Benign       357
Malignant    212
Name: count, dtype: int64

Porcentajes:
diagnosis
Benign       62.741652
Malignant    37.258348
Name: proportion, dtype: float64

=== INFORMACIÓN DE VARIABLES ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null

# --------------------------------------------------------------------------

## 3. Preprocesamiento
 - Escalamiento de características (StandardScaler)
 - División del conjunto en entrenamiento y prueba
 - usar hold-out de 70% y 30%



In [43]:
# 3. Preprocesamiento
# Separar características (X) y variable objetivo (y)
X = df.drop(['target', 'diagnosis'], axis=1)  # Eliminar columnas target y diagnosis
y = df['target']  # Usar solo la columna target numérica

print("=== DIVISIÓN TRAIN-TEST (70%-30%) ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y  # Mantener proporción de clases en train y test
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")
print(f"Distribución en train - Malignant: {sum(y_train == 0)}, Benign: {sum(y_train == 1)}")
print(f"Distribución en test - Malignant: {sum(y_test == 0)}, Benign: {sum(y_test == 1)}")

print("\n=== ESCALAMIENTO CON STANDARDSCALER ===")
# Crear el escalador
scaler = StandardScaler()

# Ajustar el escalador SOLO con los datos de entrenamiento y transformarlos
X_train_scaled = scaler.fit_transform(X_train)

# Transformar los datos de prueba usando el escalador ajustado con train
X_test_scaled = scaler.transform(X_test)

print("Escalamiento completado:")
print(f"X_train escalado - Media: {X_train_scaled.mean():.2f}, Desviación: {X_train_scaled.std():.2f}")
print(f"X_test escalado - Media: {X_test_scaled.mean():.2f}, Desviación: {X_test_scaled.std():.2f}")

# Verificar algunas características escaladas
print("\n=== EJEMPLO DE CARACTERÍSTICAS ESCALADAS ===")
print("Primeras 5 filas de X_train escalado (primeras 3 características):")
print(X_train_scaled[:5, :3])

=== DIVISIÓN TRAIN-TEST (70%-30%) ===
Tamaño del conjunto de entrenamiento: (398, 30)
Tamaño del conjunto de prueba: (171, 30)
Distribución en train - Malignant: 148, Benign: 250
Distribución en test - Malignant: 64, Benign: 107

=== ESCALAMIENTO CON STANDARDSCALER ===
Escalamiento completado:
X_train escalado - Media: -0.00, Desviación: 1.00
X_test escalado - Media: 0.03, Desviación: 1.03

=== EJEMPLO DE CARACTERÍSTICAS ESCALADAS ===
Primeras 5 filas de X_train escalado (primeras 3 características):
[[-0.70982078 -0.258417   -0.63739619]
 [-0.83033136  2.2311266  -0.87497994]
 [-1.01109725 -0.22726989 -1.03517213]
 [-0.38272061 -0.11158065 -0.40896629]
 [-0.80450767 -1.4019607  -0.8100709 ]]


# --------------------------------------------------------------------------

## 4. Aplicar técnica CNN (Condensed Nearest Neighbor)

- Aplicar CNN sobre el conjunto de entrenamiento
- Mostrar el tamaño del conjunto reducido



In [44]:
# Reshape para CNN
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# API FUNCIONAL - la única que permite model.input
inputs = Input(shape=(30, 1))
x = Conv1D(32, 3, activation='relu')(inputs)
x = MaxPooling1D(2)(x)
x = Conv1D(64, 3, activation='relu')(x)
x = MaxPooling1D(2)(x)
x = Flatten()(x)
x = Dense(50, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar
history = model.fit(X_train_cnn, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

# Evaluar
test_loss, test_accuracy = model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"Accuracy de la CNN: {test_accuracy:.4f}")

# AHORA SÍ FUNCIONA - porque usamos API Funcional
feature_extractor = Model(inputs=model.input, outputs=model.layers[-3].output)
y_train_cnn_reduced = y_train
X_train_cnn_reduced = feature_extractor.predict(X_train_cnn)
X_test_cnn_reduced = feature_extractor.predict(X_test_cnn)

print(f"Tamaño reducido CNN: {X_train_cnn_reduced.shape}")

Accuracy de la CNN: 0.9591
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Tamaño reducido CNN: (398, 384)


 --------------------------------------------------------------------------

## 5. Aplicar técnica ENN (Edited Nearest Neighbor)

- Aplicar ENN sobre el conjunto de entrenamiento
- Mostrar el tamaño del conjunto reducido



In [45]:
# 5. Aplicar técnica ENN (Edited Nearest Neighbor)
enn = EditedNearestNeighbours(
    n_neighbors=3,
    kind_sel='all'
)

X_train_enn, y_train_enn = enn.fit_resample(X_train_scaled, y_train)

print("Tamaños originales vs después de ENN:")
print(f"Original - X_train: {X_train_scaled.shape}, y_train: {y_train.shape}")
print(f"Después ENN - X_train: {X_train_enn.shape}, y_train: {y_train_enn.shape}")

# Reducción aplicada:
reduccion = (1 - len(X_train_enn) / len(X_train_scaled)) * 100
print(f"Reducción del {reduccion:.1f}% en el conjunto de entrenamiento")

# Ahora aplicar reshape para CNN
X_train_enn_cnn = X_train_enn.reshape(X_train_enn.shape[0], X_train_enn.shape[1], 1)
print(f"Tamaño final para CNN: {X_train_enn_cnn.shape}")

# GUARDAR VARIABLES PARA SVM
X_train_enn_reduced_svm = X_train_enn  # Datos reducidos por ENN para SVM
y_train_enn_reduced_svm = y_train_enn  # Targets correspondientes
X_test_enn_reduced_svm = X_test_scaled  # Test se mantiene igual

print("Variables ENN guardadas:")
print(f"X_train_enn_reduced_svm: {X_train_enn_reduced_svm.shape}")
print(f"y_train_enn_reduced_svm: {y_train_enn_reduced_svm.shape}")
print(f"X_test_enn_reduced_svm: {X_test_enn_reduced_svm.shape}")

Tamaños originales vs después de ENN:
Original - X_train: (398, 30), y_train: (398,)
Después ENN - X_train: (376, 30), y_train: (376,)
Reducción del 5.5% en el conjunto de entrenamiento
Tamaño final para CNN: (376, 30, 1)
Variables ENN guardadas:
X_train_enn_reduced_svm: (376, 30)
y_train_enn_reduced_svm: (376,)
X_test_enn_reduced_svm: (171, 30)


# --------------------------------------------------------------------------

## 6. Aplicar reducción mediante K-medias
- Realizar agrupamiento por clase y representar cada grupo con su centroide. Elige la mitad de los elementos de cada clase como el valor del número de centroides
- Generar un nuevo conjunto reducido con los centroides como prototipos



In [46]:
# 6. Aplicar reducción mediante K-medias
X_train_malignant = X_train_scaled[y_train == 0]
X_train_benign = X_train_scaled[y_train == 1]

print(f"Tamaños originales por clase:")
print(f"Malignant: {X_train_malignant.shape}")
print(f"Benign: {X_train_benign.shape}")

# Calcular número de centroides (mitad de elementos de cada clase)
n_centroides_malignant = len(X_train_malignant) // 2
n_centroides_benign = len(X_train_benign) // 2

print(f"\nNúmero de centroides por clase:")
print(f"Malignant: {n_centroides_malignant} centroides")
print(f"Benign: {n_centroides_benign} centroides")

# Para clase Malignant
kmeans_malignant = KMeans(n_clusters=n_centroides_malignant, random_state=42)
kmeans_malignant.fit(X_train_malignant)
centroides_malignant = kmeans_malignant.cluster_centers_

# Para clase Benign
kmeans_benign = KMeans(n_clusters=n_centroides_benign, random_state=42)
kmeans_benign.fit(X_train_benign)
centroides_benign = kmeans_benign.cluster_centers_

print(f"Centroides Malignant: {centroides_malignant.shape}")
print(f"Centroides Benign: {centroides_benign.shape}")

# Crear nuevo conjunto reducido con los centroides como prototipos
X_train_reducido = np.vstack([centroides_malignant, centroides_benign])
y_train_reducido = np.hstack([
    np.zeros(len(centroides_malignant)),
    np.ones(len(centroides_benign))
])

print(f"\n=== NUEVO CONJUNTO REDUCIDO ===")
print(f"Tamaño del conjunto original: {X_train_scaled.shape}")
print(f"Tamaño del conjunto reducido: {X_train_reducido.shape}")

reduccion_porcentaje = (1 - len(X_train_reducido) / len(X_train_scaled)) * 100
print(f"Reducción del {reduccion_porcentaje:.1f}%")

print(f"\nDistribución en conjunto reducido:")
print(f"Malignant: {sum(y_train_reducido == 0)} centroides")
print(f"Benign: {sum(y_train_reducido == 1)} centroides")

# Verificar algunos centroides
print(f"\n=== EJEMPLO DE CENTROIDES ===")
print("Primeros 3 centroides Malignant (primeras 5 características):")
print(centroides_malignant[:3, :5])
print("\nPrimeros 3 centroides Benign (primeras 5 características):")
print(centroides_benign[:3, :5])

# GUARDAR VARIABLES PARA SVM
X_train_kmeans_reduced_svm = X_train_reducido  # Datos reducidos por K-means para SVM
y_train_kmeans_reduced_svm = y_train_reducido  # Targets correspondientes
X_test_kmeans_reduced_svm = X_test_scaled      # Test se mantiene igual

print("\n=== VARIABLES K-MEANS GUARDADAS ===")
print(f"X_train_kmeans_reduced_svm: {X_train_kmeans_reduced_svm.shape}")
print(f"y_train_kmeans_reduced_svm: {y_train_kmeans_reduced_svm.shape}")
print(f"X_test_kmeans_reduced_svm: {X_test_kmeans_reduced_svm.shape}")

Tamaños originales por clase:
Malignant: (148, 30)
Benign: (250, 30)

Número de centroides por clase:
Malignant: 74 centroides
Benign: 125 centroides
Centroides Malignant: (74, 30)
Centroides Benign: (125, 30)

=== NUEVO CONJUNTO REDUCIDO ===
Tamaño del conjunto original: (398, 30)
Tamaño del conjunto reducido: (199, 30)
Reducción del 50.0%

Distribución en conjunto reducido:
Malignant: 74 centroides
Benign: 125 centroides

=== EJEMPLO DE CENTROIDES ===
Primeros 3 centroides Malignant (primeras 5 características):
[[ 1.56553391  0.86065398  1.54787497  1.53947044  0.39083484]
 [ 0.6157957  -0.1271542   0.7115469   0.45404715  0.81086729]
 [ 2.08200786  1.13096636  2.15743747  2.08380037  1.47091827]]

Primeros 3 centroides Benign (primeras 5 características):
[[ 0.24565603 -0.1271542   0.25551925  0.11391451 -1.08077884]
 [-0.46736495 -1.32631774 -0.47325121 -0.51721397  1.19339683]
 [-0.72539697  0.05082926 -0.75057094 -0.69993228 -0.75332497]]

=== VARIABLES K-MEANS GUARDADAS ===
X_t

 --------------------------------------------------------------------------

##7. Entrenar SVM sobre cada conjunto reducido
- Entrenar un modelo SVM (SVC) sobre:
 * los datos originales
 * datos reducidos con CNN
 * datos reducidos con ENN
 * datos reducidos con K-medias
- Evaluar cada modelo con accuracy, F1-score



In [47]:
# 7. Entrenar SVM sobre cada conjunto reducido
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

print("=== ENTRENAMIENTO Y EVALUACIÓN SVM ===")
print("=" * 50)

# Diccionario con todos los conjuntos de entrenamiento
conjuntos_entrenamiento = {
    'Original': (X_train_scaled, y_train, X_test_scaled),
    'CNN': (X_train_cnn_reduced, y_train_cnn_reduced, X_test_cnn_reduced),
    'ENN': (X_train_enn_reduced_svm, y_train_enn_reduced_svm, X_test_enn_reduced_svm),
    'K-means': (X_train_kmeans_reduced_svm, y_train_kmeans_reduced_svm, X_test_kmeans_reduced_svm)
}
# Diccionario para almacenar resultados
resultados_svm = {}

for nombre, (X_train_set, y_train_set,X_test_set) in conjuntos_entrenamiento.items():
    print(f"\n--- Entrenando SVM con {nombre} ---")
    print(f"Tamaño entrenamiento: {X_train_set.shape}")
    print(f"Tamaño prueba: {X_test_set.shape}")
    print(f"Distribución clases - Malignant: {sum(y_train_set == 0)}, Benign: {sum(y_train_set == 1)}")
    # Entrenar modelo SVM
    svm_model = SVC(kernel='rbf', random_state=42)
    svm_model.fit(X_train_set, y_train_set)

    # Predecir en conjunto de prueba
    y_pred = svm_model.predict(X_test_set)

    # Calcular métricas
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Guardar resultados
    resultados_svm[nombre] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'tamaño_entrenamiento': X_train_set.shape[0],
        'modelo': svm_model
    }

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score: {f1:.4f}")

# Mostrar comparativa final
print("\n" + "=" * 60)
print("COMPARATIVA FINAL - SVM SOBRE DIFERENTES CONJUNTOS")
print("=" * 60)

print(f"\n{'Método':<12} {'Tamaño Train':<14} {'Accuracy':<10} {'F1-Score':<10}")
print("-" * 50)

for metodo, metrics in resultados_svm.items():
    print(f"{metodo:<12} {metrics['tamaño_entrenamiento']:<14} {metrics['accuracy']:.4f}     {metrics['f1_score']:.4f}")

# Identificar el mejor método
mejor_accuracy = max(resultados_svm.items(), key=lambda x: x[1]['accuracy'])
mejor_f1 = max(resultados_svm.items(), key=lambda x: x[1]['f1_score'])

print(f"\n=== MEJORES RESULTADOS ===")
print(f"Mejor Accuracy: {mejor_accuracy[0]} - {mejor_accuracy[1]['accuracy']:.4f}")
print(f"Mejor F1-Score: {mejor_f1[0]} - {mejor_f1[1]['f1_score']:.4f}")

# Análisis de reducción vs performance
print(f"\n=== ANÁLISIS DE REDUCCIÓN ===")
tamaño_original = resultados_svm['Original']['tamaño_entrenamiento']
for metodo, metrics in resultados_svm.items():
    if metodo != 'Original':
        reduccion = (1 - metrics['tamaño_entrenamiento'] / tamaño_original) * 100
        perdida_accuracy = (resultados_svm['Original']['accuracy'] - metrics['accuracy']) * 100
        print(f"{metodo}: Reducción {reduccion:.1f}% - Pérdida accuracy: {perdida_accuracy:.2f}%")

=== ENTRENAMIENTO Y EVALUACIÓN SVM ===

--- Entrenando SVM con Original ---
Tamaño entrenamiento: (398, 30)
Tamaño prueba: (171, 30)
Distribución clases - Malignant: 148, Benign: 250
Accuracy: 0.9766
F1-Score: 0.9813

--- Entrenando SVM con CNN ---
Tamaño entrenamiento: (398, 384)
Tamaño prueba: (171, 384)
Distribución clases - Malignant: 148, Benign: 250
Accuracy: 0.9649
F1-Score: 0.9720

--- Entrenando SVM con ENN ---
Tamaño entrenamiento: (376, 30)
Tamaño prueba: (171, 30)
Distribución clases - Malignant: 148, Benign: 228
Accuracy: 0.9708
F1-Score: 0.9763

--- Entrenando SVM con K-means ---
Tamaño entrenamiento: (199, 30)
Tamaño prueba: (171, 30)
Distribución clases - Malignant: 74, Benign: 125
Accuracy: 0.9766
F1-Score: 0.9817

COMPARATIVA FINAL - SVM SOBRE DIFERENTES CONJUNTOS

Método       Tamaño Train   Accuracy   F1-Score  
--------------------------------------------------
Original     398            0.9766     0.9813
CNN          398            0.9649     0.9720
ENN          

# --------------------------------------------------------------------------

## 📊 8. Comparar los resultados y reflexión final
- **Comparar las métricas de rendimiento obtenidas con cada técnica**

*K-means:* 50% reducción, mismo accuracy, MEJOR F1-score

*Original:* Buen desempeño, menos eficiente

*ENN:* Reducción moderada, pequeña pérdida

*CNN:* Sin beneficios, mayor complejidad

- **Escribe tus conclusiones sobre el impacto de la reducción de la numerosidad**

Más datos no significa mejor modelo y se bede de encontrar el punto exacto ya que al reducir 5% los datos si se perdio 0.5% de accuracy.

- **¿Cuál técnica funcionó mejor? ¿Qué ventajas y desventajas observaste?**

K-means fue la técnica más efectiva para este problema específico, demostrando que una reducción inteligente de datos puede mantener el desempeño mientras mejora significativamente la eficiencia computacional.
